# Heritage Short: ACE-Step 1.5 score candidates (verified on Kaggle 2× T4, 2026-11-25)

Generates one 75 s instrumental per seed in `music_job/job.json`. Measured: about 60 s per candidate on a T4 (COSTS.md).

**Run order:** cells 1 → 4. Settings: GPU T4, Internet on, the job bundle attached as a dataset.

**Why it looks like this (FAILURES.md F-25):**
- ACE-Step gets its **own Python 3.11 venv**. Installing it into Kaggle's system Python broke numpy and jax.
- Every install names the venv with `--python`. Kaggle's `uv` otherwise targets the system Python.
- The torch trio is pinned together.
- flash-attn is skipped; it can't build on a T4.

In [ ]:
# Common setup: job files, output folder, helpers. Rerun this cell after any kernel restart.
import glob, json, os, subprocess, sys, time, datetime, shutil, re
from pathlib import Path
os.environ["MPLBACKEND"] = "Agg"   # subprocesses inherit the notebook's inline backend otherwise (F-25)

jobs = sorted(glob.glob("/kaggle/input/**/job.json", recursive=True))
def job(stage): return Path(next(j for j in jobs if json.load(open(j)).get("stage") == stage))
OUT = Path("/kaggle/working/kaggle_results")

def run(cmd, **kw):                 # stream a command; raise on failure
    print("$", cmd if isinstance(cmd, str) else " ".join(map(str, cmd)), flush=True)
    subprocess.run(cmd, shell=isinstance(cmd, str), check=True, **kw)

def log(folder, stage, shot, model, params, seed, status, latency, output=None, error=None):
    rec = {"ts": datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
           "run_id": "kaggle-" + stage, "stage": stage, "shot": shot, "provider": "kaggle", "model": model,
           "params": params, "seed": seed, "input_hash": None, "status": status, "attempt": 1,
           "latency_s": round(latency, 3), "gpu_s": round(latency, 3), "cost_usd_list": None,
           "output": output, "error": str(error)[-500:] if error else None}
    with open(folder / "runs_fragment.jsonl", "a") as f: f.write(json.dumps(rec, ensure_ascii=False) + "\n")

run("nvidia-smi --query-gpu=index,name,memory.total --format=csv")
run(f"{sys.executable} -m pip install -q uv")

MUSIC_JOB = job("music"); MUSIC = json.loads(MUSIC_JOB.read_text())
assert len(MUSIC["outs"]) == MUSIC["candidates"]
(OUT / "music").mkdir(parents=True, exist_ok=True)
print(MUSIC["outs"])

In [ ]:
# 2. ACE-Step venv + checkpoints + smoke test (about 5-10 min)
ACE, ACE_PY = "/kaggle/working/ACE-Step-1.5", "/tmp/ace/bin/python"
run(f"test -d {ACE} || git clone -q https://github.com/ACE-Step/ACE-Step-1.5.git {ACE}")
run("uv venv --clear /tmp/ace --python 3.11")        # ACE-Step requires Python >=3.11,<3.13
U = f"uv pip install --python {ACE_PY}"
TRIO = "torch==2.10.0 torchvision==0.25.0 torchaudio==2.10.0"   # must match each other (F-25)
run(f"{U} {TRIO} --index-url https://download.pytorch.org/whl/cu128")
run(f"grep -viE '^(flash.?attn|torch|torchvision|torchaudio|triton-windows)([<>=~ ;+].*)?$' {ACE}/requirements.txt "
    f"| grep -v '^#' | grep -v '^-e' > /tmp/ace_req.txt")
run(f"{U} -r /tmp/ace_req.txt diskcache typer-slim pytorch-wavelets pywavelets 'setuptools<72'")
run(f"{U} {TRIO} 'torchcodec==0.10.*' --index-url https://download.pytorch.org/whl/cu128 "
    f"--extra-index-url https://pypi.org/simple")      # re-pin: requirements may pull a newer torchaudio/torchcodec
run(f"{U} --no-deps -e {ACE}")
run(f"cd {ACE} && /tmp/ace/bin/acestep-download")     # DiT, VAE, text encoder, 5Hz LM -> ACE-Step-1.5/checkpoints
run([ACE_PY, "-c", """
import sys; sys.path.insert(0, '/kaggle/working/ACE-Step-1.5')
import torch, torchaudio, loguru, soundfile
assert sys.prefix == '/tmp/ace', sys.prefix
assert torch.cuda.is_available(); x = torch.randn(512, 512, device='cuda'); (x @ x).sum().item()
from acestep.handler import AceStepHandler
from acestep.llm_inference import LLMHandler
from acestep.inference import GenerationParams, GenerationConfig, generate_music
print('ACE OK  torch', torch.__version__, torch.cuda.get_device_name(0))
"""])

In [ ]:
# 3. Generate the candidates (Python API, as in the repo's run_generate_test.py)
RUNNER = r'''
import sys, os, json, time, glob, shutil, datetime
from pathlib import Path
ACE = "/kaggle/working/ACE-Step-1.5"; sys.path.insert(0, ACE); os.chdir(ACE)
CK = os.path.join(ACE, "checkpoints")
job = json.loads(Path(sys.argv[1]).read_text()); out = Path(sys.argv[2])
def log(shot, seed, status, latency, output=None, error=None):
    rec = {"ts": datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"), "run_id": "kaggle-music",
           "stage": "music", "shot": shot, "provider": "kaggle", "model": job["model"],
           "params": {k: job[k] for k in ("duration_s", "bpm", "key", "caption")}, "seed": seed,
           "input_hash": None, "status": status, "attempt": 1, "latency_s": round(latency, 3),
           "gpu_s": round(latency, 3), "cost_usd_list": None, "output": output,
           "error": str(error)[-500:] if error else None}
    with open(out / "runs_fragment.jsonl", "a") as f: f.write(json.dumps(rec, ensure_ascii=False) + "\n")

import torch
from acestep.handler import AceStepHandler
from acestep.llm_inference import LLMHandler
from acestep.inference import GenerationParams, GenerationConfig, generate_music
dit = AceStepHandler()
msg, ok = dit.initialize_service(project_root=ACE, config_path="acestep-v15-turbo", device="cuda", offload_to_cpu=False)
print("DiT:", ok, msg, flush=True)
if not ok: sys.exit(1)
llm = LLMHandler()
lm_name = next((d for d in ("acestep-5Hz-lm-0.6B", "acestep-5Hz-lm-1.7B") if os.path.isdir(os.path.join(CK, d))), None)
use_lm = False
if lm_name:
    msg, ok = llm.initialize(checkpoint_dir=CK, lm_model_path=lm_name, backend="pt",
                             device="cuda", offload_to_cpu=True, dtype=None)
    print("LM:", ok, msg, flush=True); use_lm = bool(ok)

n = len(job["outs"]); tmp = out / "_tmp"
for i, (seed, name) in enumerate(zip(job["seeds"], job["outs"]), 1):
    target = out / name
    if target.exists(): print(f"[{i}/{n}] {name} exists, skipped", flush=True); continue
    t0 = time.time()
    try:
        shutil.rmtree(tmp, ignore_errors=True); tmp.mkdir(parents=True)
        p = GenerationParams(task_type="text2music", caption=job["caption"], lyrics="[Instrumental]", instrumental=True,
                             bpm=job["bpm"], keyscale=job["key"], timesignature="4", vocal_language="unknown",
                             duration=float(job["duration_s"]), inference_steps=8, guidance_scale=1.0,
                             seed=seed, thinking=use_lm)
        cfg = GenerationConfig(batch_size=1, use_random_seed=False, seeds=[seed], audio_format="wav")
        r = generate_music(dit, llm if use_lm else None, params=p, config=cfg, save_dir=str(tmp))
        if not r.success: raise RuntimeError(r.error or r.status_message)
        paths = [a.get("path") for a in r.audios if a.get("path")] or glob.glob(str(tmp / "**" / "*.wav"), recursive=True)
        if not paths: raise RuntimeError("no audio written")
        shutil.move(paths[0], target)
        log(f"c{seed}", seed, "ok", time.time() - t0, name)
        print(f"[{i}/{n}] {name} ok  {time.time()-t0:.0f}s", flush=True)
    except Exception as e:
        log(f"c{seed}", seed, "error", time.time() - t0, None, e)
        print(f"[{i}/{n}] {name} FAILED: {str(e)[-800:]}", flush=True)
    finally:
        torch.cuda.empty_cache()
shutil.rmtree(tmp, ignore_errors=True)
'''
Path("/kaggle/working/run_ace.py").write_text(RUNNER)
subprocess.run([ACE_PY, "/kaggle/working/run_ace.py", str(MUSIC_JOB), str(OUT / "music")],
               env={**os.environ, "CUDA_VISIBLE_DEVICES": "0"})
run(f"ls -la {OUT/'music'}")

In [ ]:
# 4. Zip for download, then locally: py -m pipeline music --import outputs/kaggle_results/music
shutil.make_archive("/kaggle/working/music_results", "zip", OUT / "music")
print("Download /kaggle/working/music_results.zip from the Output panel")